# Локальное обучение на Apple MPS

Реальные предобученные ResNet18 и ArcFace; 1000 лекционных позитивов, 5000 FairFace и дополнение 14+/8− до аудита. Обучение и генерация строго на MPS, без CPU fallback. SCRFD и работа с файлами используют CPU.

Запуск из корня проекта: `.venv/bin/python tools/run_local_notebook.py`. Выполненный ноутбук сохраняется в `work/local-mps/local_train_mps_v3.executed.ipynb`, результаты — `outputs/runs/domain-mps-v3`. При повторном запуске завершённые модели пропускаются, обучение продолжается с `last.pt`. Ноутбук не публикует файлы на GitHub.

Случайное разбиение может включать похожие кадры из одной лекции в разных частях выборки; такой test не доказывает качество на новой лекции.
Подготовлены новые код и данные; обучение этой версии ещё не выполнялось. Перед ручным запуском восстановите дополнение командой из docs/DOMAIN_REPAIR.md.


In [ ]:
import os, sys, gc, json
from pathlib import Path
ROOT = Path.cwd()
assert (ROOT / 'src/sokolov').is_dir(), 'Запустите ноутбук из корня репозитория'
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '0'
os.environ['TORCH_HOME'] = str(ROOT/'work/torch-cache')
os.environ['KAGGLEHUB_CACHE'] = str(ROOT/'work/kaggle-cache')
os.environ['MPLCONFIGDIR'] = str(ROOT/'work/matplotlib')
os.environ['NO_ALBUMENTATIONS_UPDATE'] = '1'
sys.path.insert(0, str(ROOT/'src'))
import torch
from dataclasses import asdict
from sokolov.config import load_config
from sokolov.model import build_model, set_training_stage, ensure_arcface_checkpoint
from sokolov.io import write_json, read_json, environment
assert torch.backends.mps.is_available(), 'MPS недоступен — CPU fallback запрещён'
assert torch.ones(1, device='mps').device.type == 'mps'
configs = [load_config(f'configs/{name}_local_mps.toml') for name in ('resnet18','arcface')]
assert all(c.training.device == 'mps' for c in configs)
ensure_arcface_checkpoint(configs[1].model)
# Проверяем чтение весов, forward и обратный проход по пикселям на MPS.
for config in configs:
    model = build_model(config.model).to('mps').eval().requires_grad_(False)
    pixels = torch.rand(1, 3, config.model.image_size, config.model.image_size, device='mps', requires_grad=True)
    logits = model(pixels)
    logits.sum().backward()
    assert logits.device.type == pixels.grad.device.type == 'mps'
    assert torch.isfinite(pixels.grad).all() and pixels.grad.abs().sum().item() > 0
    print(config.model.architecture, 'реальные веса, forward/backward: MPS OK')
    del model, pixels, logits
    gc.collect(); torch.mps.empty_cache()
RUN = Path(os.environ.get('SOKOLOV_RUN_DIR', str(ROOT/'outputs/runs/domain-mps-v3')))
RUN.mkdir(parents=True, exist_ok=True)
write_json(RUN/'environment.json', {**environment(), 'device':'mps', 'fallback':False})
write_json(RUN/'status.json', {'status':'running'})
print('PyTorch', torch.__version__, '· MPS · CPU fallback: OFF')

In [ ]:
from sokolov.data.download import download_data
# Скачанные данные уже находятся в локальном кеше. Никакие изображения не отправляются наружу.
raw = download_data(configs[0])
print('Исходные данные:', raw)
print(read_json(raw.parent/'selection.local.json')['counts'])

In [ ]:
from sokolov.data.prepare import prepare
prepared = prepare(configs[0])
print(json.dumps(read_json(prepared.parent/'data_audit.json')['counts'], ensure_ascii=False, indent=2))

In [ ]:
from sokolov.train import train
config = configs[0]
folder = RUN/'resnet18'
state = read_json(folder/'status.json')['status'] if (folder/'status.json').exists() else None
if state not in ('trained','evaluated'):
    train(config, RUN, resume=(folder/'last.pt').exists())
else:
    print('resnet18: обучение уже завершено')
gc.collect(); torch.mps.empty_cache()

In [ ]:
from sokolov.evaluate import evaluate_test
artifacts = evaluate_test(configs[0], RUN/'resnet18'/'best.pt')
print('resnet18:', read_json(RUN/'resnet18'/'test_metrics.json'))
gc.collect(); torch.mps.empty_cache()

In [ ]:
from sokolov.train import train
config = configs[1]
folder = RUN/'arcface'
state = read_json(folder/'status.json')['status'] if (folder/'status.json').exists() else None
if state not in ('trained','evaluated'):
    train(config, RUN, resume=(folder/'last.pt').exists())
else:
    print('arcface: обучение уже завершено')
gc.collect(); torch.mps.empty_cache()

In [ ]:
from sokolov.evaluate import evaluate_test
artifacts = evaluate_test(configs[1], RUN/'arcface'/'best.pt')
print('arcface:', read_json(RUN/'arcface'/'test_metrics.json'))
gc.collect(); torch.mps.empty_cache()

In [ ]:
from sokolov.synthesize import select_best_model, generate_best
best = select_best_model(RUN)
print('Сохранена лучшая модель:', best)
print(read_json(RUN/'best_model.json'))
generate_best(RUN, device='mps')
gc.collect(); torch.mps.empty_cache()

In [ ]:
from sokolov.report import make_report
import shutil
report = make_report(RUN)
archive = shutil.make_archive(str(RUN/'report'), 'zip', root_dir=RUN, base_dir='report')
print('Отчёт:', report/'report.html')
print('Архив:', archive)
from IPython.display import display, HTML
display(HTML((report/'report.html').read_text()))
write_json(RUN/'status.json', {'status':'completed'})
print('Проверка фото: python tools/serve_ui.py --run-dir', RUN)